<strong><span style="color:skyblue;font-size:40px"> Reinforcement Learning model for Quant analysis</strong><br>
- Data source from yahoo finance<br>
- train and test timer periods<br>
- Gymnasium for the model reward<br>

In [2]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import pandas as pd
import stable_baselines3 as sb3
import yfinance as yf
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.monitor import Monitor
import os


<strong><span style="color:lightgreen;font-size:30px">Agent environment Build</strong><br>
- Uses OpenAI's gymnasium space<br>
<span style="color:red">- Reward function requires work!</strong>

In [3]:
class StockTradingEnv(gym.Env):

    def __init__(self, data):
        self.data = data
        self.current_step = 0
        self.portfolio_history = []
        self.cash = 10000
        self.holdings = 0
        
        # ✅ NEW: initialize action and portfolio value logs
        self.action_history = []
        self.portfolio_value_history = []
        
        for col in self.data.columns:
            self.data[col] = pd.to_numeric(self.data[col], errors='coerce')

        self.action_space = spaces.Discrete(3)  # Buy, Hold, Sell
        self.numeric_cols = self.data.select_dtypes(include=np.number).columns
        obs_shape = len(self._get_state())
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(obs_shape,), dtype=np.float32)
        self.reset()

    def _get_state(self):
        row = self.data.iloc[self.current_step][self.numeric_cols]
        cleaned = row.fillna(0).replace([np.inf, -np.inf], 0).astype(np.float32)  # 🧼 Explicit cast
        state = np.concatenate([cleaned.values, [self.cash, self.holdings]], axis=0)
        return state
    
    def step(self, action):
         # ✅ NEW: log action at current step
        self.action_history.append(action)

        # ✅ NEW: log portfolio value before action (you can also log after, if you prefer)
        portfolio_value = self.cash + self.holdings * self.data.iloc[self.current_step]['Close']
        self.portfolio_value_history.append(portfolio_value)
        
        # Reward based on the current action
        reward = self.calculate_reward(action)
        # Advance to the next time step
        self.current_step += 1
        # Check if we’re done (end of the data)
        terminated = self.current_step >= len(self.data) - 1
        truncated = False  # Could be used for time limits, etc.
        # Get the next state observation
        observation = self._get_state()
        # Info dict — can be used for debugging, analytics, etc.
        info = {"portfolio_value": self.cash + self.holdings * self.data.iloc[self.current_step]['Close'],}
        return observation, reward, terminated, truncated, info

# Conservative reward

    # def calculate_reward(self, action):
    #     """
    #     Reward based on:
    #     - Net change in portfolio value
    #     - Transaction costs for buying/selling
    #     - Penalty for unbalanced allocation
    #     - Optional volatility penalty for unstable portfolio value
    #     """

    #     price = self.data.iloc[self.current_step]['Close']
    #     transaction_cost = 0.001
    #     cost = price * transaction_cost

    #     prev_value = self.cash + self.holdings * price

    #     if action == 1 and self.cash >= price + cost:
    #         self.holdings += 1
    #         self.cash -= price + cost
    #     elif action == 2 and self.holdings > 0:
    #         self.holdings -= 1
    #         self.cash += price - cost

    #     new_value = self.cash + self.holdings * price
    #     reward = (new_value - prev_value) / (prev_value + 1e-8)

    #     # ✅ Gentle encouragement to hold *some* stock
    #     allocation = self.holdings * price / (new_value + 1e-8)
    #     allocation_bonus = 1 - abs(allocation - 0.5)  # peak at 0.5
    #     reward += 0.1 * allocation_bonus  # small boost

    #     # ✅ Optional: volatility penalty
    #     self.portfolio_history.append(new_value)
    #     if len(self.portfolio_history) > 10:
    #         recent_vol = np.std(self.portfolio_history[-10:])
    #         reward -= 0.05 * recent_vol  # slightly less punishing

    #     return np.clip(reward, -100, 100)

# aggressive reward

    def calculate_reward(self, action):
        """
        Reward function for a high-risk, high-signal model.
        Penalises bad trades harshly, amplifies good trades.
        """

        price = self.data.iloc[self.current_step]['Close']
        prev_price = self.data.iloc[self.current_step - 1]['Close'] if self.current_step > 0 else price
        transaction_cost = 0.001
        cost = price * transaction_cost

        prev_value = self.cash + self.holdings * price

        # 💸 Execute action
        if action == 1 and self.cash >= price + cost:  # Buy
            self.holdings += 1
            self.cash -= price + cost
        elif action == 2 and self.holdings > 0:  # Sell
            self.holdings -= 1
            self.cash += price - cost

        new_value = self.cash + self.holdings * price

        # 📈 Base reward: portfolio change
        reward = (new_value - prev_value) / (prev_value + 1e-8) * 100  # 💥 Amplify signal

        # ⚖️ Allocation bonus (balance OR bold, but not chaos)
        allocation = self.holdings * price / (new_value + 1e-8)
        reward += 20 * allocation * (1 - abs(allocation - 0.5))  # peak at 50% allocation

        # 🚨 Action-based logic
        # 💡 Buying into a rise? Good!
        if action == 1 and price > prev_price:
            reward += 15
        # 💰 Selling at a profit? Nice.
        if action == 2 and price > prev_price:
            reward += 15
        # 😬 Buy into falling price? Dumb.
        if action == 1 and price < prev_price:
            reward -= 10
        # 😱 Sell before price surge? Ouch.
        if action == 2 and price < prev_price:
            reward -= 10
        # 🛌 Holding in a volatile swing? Lazy.
        if action == 0 and abs(price - prev_price) > 0.01 * price:
            reward -= 5
        # 🔍 Debug prints
        if abs(reward) > 5:
            print(f"[Step {self.current_step}] Reward: {reward:.2f}")
        return reward

    def reset(self, *, seed=None, options=None):
        # ✅ NEW: reset logs at beginning of each episode
        self.action_history = []
        self.portfolio_value_history = []
        
        self.current_step = 0
        self.cash = 10000
        self.holdings = 0
        if seed is not None:
            np.random.seed(seed)    
        return self._get_state(), {}


Financial moddeling functions

In [4]:
def rolling_statistics(data, window):
    """
    Calculates rolling statistics given a series of data and window size.
    Returns: Rolling Max, Min, Standard deviation, Average
    Parameters:
        data: Pandas Series
        window: int, length of rolling window
    """
    rolling_max = data.rolling(window=window).max()
    rolling_min = data.rolling(window=window).min()
    rolling_std = data.rolling(window=window).std()
    rolling_average = data.rolling(window=window).mean()
    return rolling_max, rolling_min, rolling_std, rolling_average

def rsi (data,window):
    """
    calcualtes the RSI score (Relative Strength Index) for a given data series
    :paramaters
        data: pandas Series
        Window: int, length of rolling window
    """
    delta = data.diff()
    gain = delta.where(delta > 0, 0)
    loss = -delta.where(delta > 0, 0)
    avg_gain = gain.rolling(window=window).mean()
    avg_loss = loss.rolling(window=window).mean()
    rs = avg_gain/(avg_loss + 1e-10)  # Avoid division by zero
    rsi = 100 - (100 / (1+rs))
    return rsi

def macd(data, short_window, long_window, signal_window):
    """
    Moving Average Convergence Divergence (MACD) calculation.
    Meansures trend strength and direction.
    Paramenters:
        data: Pandas Series
        short_window: int, short-term moving average window - recomended = 12
        long_window: int, Long-term moving average window - recomended = 26
        signal_window: int, Signal line moving average window - recomended = 9
    Retuerns:
        MACD_line:Pandas Series
        Signal_line: Pandas Series
    """
    ema_short = data.ewm(span=short_window, adjust=False).mean()
    ema_long = data.ewm(span=long_window, adjust=False).mean()
    macd_line = ema_short - ema_long
    macd_signal = macd_line.ewm(span=signal_window, adjust=False).mean()
    return macd_line, macd_signal


<strong><span style="font-size:30px;color:lightpink">download stocks data from yfinance ready for analysis</strong><br>
| Ticker  | Company Name            |
|---------|------------------------|
| TSLA    | Tesla Inc              |
| NVDA    | NVIDIA Corporation     |
| PG      | Procter & Gamble Co    |
| BARC.L  | Barclays PLC           |
| EZJ.L   | EasyJet PLC            |
| BA.L    | BAE Systems PLC        |

In [5]:
tickers = ['TSLA', 'NVDA', 'PG', 'BARC.L', 'EZJ.L', 'BA.L']
print("Downloading data stocks data...")
data = yf.download(tickers= tickers, start='2018-01-01', end='2024-12-31', group_by='ticker')
print("stocks data downloaded.")
# reshare the data to be multi idex table
data = data.stack(level=0).rename_axis(['Date', 'Ticker']).reset_index()
# convert to ordered by ticker then date
data = data.sort_values(by=['Ticker', 'Date'])
print(data.dtypes)

training_data = data[data['Date'] < '2022-01-01']
testing_data = data[data['Date'] >= '2022-01-01']

YF.download() has changed argument auto_adjust default to True


[*********************100%***********************]  6 of 6 completed

stocks data downloaded.
Price
Date      datetime64[ns]
Ticker            object
Open             float64
High             float64
Low              float64
Close            float64
Volume           float64
dtype: object



/var/folders/f2/_cjvdffn2x91f953fzbj08dc0000gn/T/ipykernel_931/321082509.py:6: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  data = data.stack(level=0).rename_axis(['Date', 'Ticker']).reset_index()


In [6]:
features = ['Open', 'High', 'Low', 'Close', 'Volume']

symbol = 'TSLA'

symbol_data = training_data[training_data['Ticker'] == symbol][features]


for feature in features:
    #calculate rolling statistics
    rolling_max, rolling_min, rolling_std, rolling_avg = rolling_statistics(symbol_data[feature], window=21)
    symbol_data[feature + '_rolling_max_21'] = rolling_max
    symbol_data[feature + '_rolling_min_21'] = rolling_min
    symbol_data[feature + '_rolling_std_21'] = rolling_std
    symbol_data[feature + '_rolling_avg_21'] = rolling_avg
    
    # Calculate RSI
    symbol_data[feature + '_rsi'] = rsi(symbol_data[feature], window=14)
    
    # Calculate MACD
    macd_line, macd_signal = macd(symbol_data[feature], short_window=12, long_window=26, signal_window=9)
    symbol_data[feature + '_macd'] = macd_line
    symbol_data[feature + '_macd_signal'] = macd_signal

display(symbol_data.head())

symbol_data = symbol_data[['Close', 'Volume', 'Close_rolling_max_21',
       'Close_rolling_min_21', 'Close_rolling_std_21', 'Close_rolling_avg_21',
       'Close_rsi', 'Close_macd', 'Close_macd_signal']].copy()

log_dir = "/Users/zacwells/Desktop/Reinforcement Learing.stock_trading_env_TSLA_PPO.csv"
os.makedirs(log_dir,exist_ok=True)

# env = StockTradingEnv(symbol_data)
# check_env(env, warn=True)

# model = sb3.PPO("MlpPolicy", env, verbose=1, ent_coef=0.05)
# model.learn(total_timesteps=2_000_000)

Price,Open,High,Low,Close,Volume,Open_rolling_max_21,Open_rolling_min_21,Open_rolling_std_21,Open_rolling_avg_21,Open_rsi,...,Close_rsi,Close_macd,Close_macd_signal,Volume_rolling_max_21,Volume_rolling_min_21,Volume_rolling_std_21,Volume_rolling_avg_21,Volume_rsi,Volume_macd,Volume_macd_signal
5,20.799999,21.474001,20.733334,21.368668,65283000.0,NaN,NaN,NaN,NaN,NaN,...,NaN,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,0.000000e+00,0.000000e+00
11,21.400000,21.683332,21.036667,21.150000,67822500.0,NaN,NaN,NaN,NaN,NaN,...,NaN,-0.017444,-0.003489,NaN,NaN,NaN,NaN,NaN,2.025812e+05,4.051624e+04
17,20.858000,21.236668,20.378668,20.974667,149194500.0,NaN,NaN,NaN,NaN,NaN,...,NaN,-0.044898,-0.011771,NaN,NaN,NaN,NaN,NaN,6.850204e+06,1.402454e+06
23,21.108000,21.149332,20.799999,21.105333,68868000.0,NaN,NaN,NaN,NaN,NaN,...,NaN,-0.055473,-0.020511,NaN,NaN,NaN,NaN,NaN,5.572578e+06,2.236479e+06
29,21.066668,22.468000,21.033333,22.427334,147891000.0,NaN,NaN,NaN,NaN,NaN,...,NaN,0.042333,-0.007942,NaN,NaN,NaN,NaN,NaN,1.081191e+07,3.951565e+06


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from stable_baselines3 import PPO, A2C
from stable_baselines3.common.callbacks import CheckpointCallback

# --- Model configurations ---
model_configs = [
    # 🧠 Baseline model
    {"name": "Vanilla PPO", "kwargs": {}},

    # 🔥 Aggressive Explorer
    {"name": "Aggressive 1", "kwargs": {
        "ent_coef": 0.2,
        "vf_coef": 0.1,
        "learning_rate": 0.0005,
        "gamma": 0.95,
        "gae_lambda": 0.8,
        "clip_range": 0.3,
        "n_steps": 512,
        "batch_size": 128,
        "n_epochs": 5,
    }},

    # 🔥 Aggressive 2: faster, dumber
    {"name": "Aggressive 2", "kwargs": {
        "ent_coef": 0.3,
        "vf_coef": 0.05,
        "learning_rate": 0.001,
        "gamma": 0.9,
        "gae_lambda": 0.7,
        "clip_range": 0.4,
        "n_steps": 256,
        "batch_size": 64,
        "n_epochs": 10,
    }},

    # 🤖 Default but nudged toward exploration
    {"name": "Curious Mind", "kwargs": {
        "ent_coef": 0.1,
        "vf_coef": 0.5,
    }},

    # 🧊 Conservative (like an actuary)
    {"name": "Conservative 1", "kwargs": {
        "ent_coef": 0.005,
        "vf_coef": 0.9,
        "learning_rate": 0.0001,
        "gamma": 0.99,
        "gae_lambda": 0.95,
        "clip_range": 0.1,
        "n_steps": 2048,
        "batch_size": 256,
        "n_epochs": 4,
    }},

    # 🧊 Conservative 2: super risk-averse
    {"name": "Conservative 2", "kwargs": {
        "ent_coef": 0.001,
        "vf_coef": 1.0,
        "learning_rate": 0.00005,
        "gamma": 0.995,
        "gae_lambda": 0.98,
        "clip_range": 0.05,
        "n_steps": 4096,
        "batch_size": 512,
        "n_epochs": 3,
    }},

    # 🎲 High entropy + balanced stability
    {"name": "Explorer Balanced", "kwargs": {
        "ent_coef": 0.15,
        "vf_coef": 0.3,
        "learning_rate": 0.0003,
        "gamma": 0.97,
        "gae_lambda": 0.85,
        "clip_range": 0.2,
        "n_steps": 1024,
        "batch_size": 128,
        "n_epochs": 6,
    }},

    # 🌀 High learning rate experiment
    {"name": "Fast Learner", "kwargs": {
        "ent_coef": 0.05,
        "vf_coef": 0.3,
        "learning_rate": 0.0015,
        "gamma": 0.98,
        "gae_lambda": 0.92,
        "clip_range": 0.2,
        "n_steps": 1024,
        "batch_size": 64,
        "n_epochs": 4,
    }},

    # 💀 YOLO Mode
    {"name": "YOLO Trader", "kwargs": {
        "ent_coef": 0.5,
        "vf_coef": 0.05,
        "learning_rate": 0.002,
        "gamma": 0.9,
        "gae_lambda": 0.7,
        "clip_range": 0.5,
        "n_steps": 128,
        "batch_size": 32,
        "n_epochs": 10,
    }},

    # 🧬 Adaptive Balance (sweet spot attempt)
    {"name": "Adaptive Balanced", "kwargs": {
        "ent_coef": 0.07,
        "vf_coef": 0.6,
        "learning_rate": 0.0003,
        "gamma": 0.985,
        "gae_lambda": 0.92,
        "clip_range": 0.15,
        "n_steps": 2048,
        "batch_size": 128,
        "n_epochs": 5,
    }},
]

checkpoints = [100_000, 500_000, 1_000_000, 1_500_000, 2_000_000]
checkpoint_dir = "./checkpoints"

os.makedirs(checkpoint_dir, exist_ok=True)

# --- Helper: Train + Save Checkpoints ---
def train_with_checkpoints(config, env_class, train_data):
    model_name = config["name"].replace(" ", "_").replace("=", "").replace(",", "")
    results = []

    env = env_class(train_data.copy())
    model = PPO("MlpPolicy", env, verbose=0, **config["kwargs"])

    for step in checkpoints:
        model_path = f"{checkpoint_dir}/{model_name}_{step}_steps.zip"

        # ✅ Skip training if checkpoint exists
        if os.path.exists(model_path):
            print(f"Skipping training for {model_name} at {step} steps (already saved)")
            results.append((step, model_path))
            continue

        print(f"Training {model_name} to {step} steps...")
        model.learn(total_timesteps=step - model.num_timesteps)
        model.save(model_path)
        results.append((step, model_path))
    return results

# --- Helper: Evaluate a model checkpoint ---
def evaluate_checkpoint(model_path, test_data, env_class, name):
    model = PPO.load(model_path)
    test_env = env_class(test_data.copy())
    obs, _ = test_env.reset()
    done = False
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, done, truncated, info = test_env.step(action)

    return {
        "name": f"{name} – {model_path.split('_')[-2]}",
        "actions": test_env.action_history,
        "portfolio": test_env.portfolio_value_history,
        "prices": test_env.data['Close'].iloc[:len(test_env.action_history)].values
    }

# --- Main Routine ---
all_results = []

for config in model_configs:
    checkpoints_info = train_with_checkpoints(config, StockTradingEnv, training_data)
    for step, model_path in checkpoints_info:
        result = evaluate_checkpoint(model_path, testing_data, StockTradingEnv, config["name"])
        all_results.append(result)

# --- Plotting ---
def plot_actions(results):
    plt.figure(figsize=(15, 6))
    plt.plot(results["prices"], label="Price", color="black")

    actions = np.array(results["actions"])
    buy_idx = np.where(actions == 1)[0]
    sell_idx = np.where(actions == 2)[0]

    plt.scatter(buy_idx, results["prices"][buy_idx], marker='^', color='green', label='Buy', s=100)
    plt.scatter(sell_idx, results["prices"][sell_idx], marker='v', color='red', label='Sell', s=100)
    plt.title(f"Trading Actions – {results['name']}")
    plt.legend()
    plt.grid(False)
    plt.tight_layout()
    plt.show()

# --- Plot all results
for result in all_results:
    plot_actions(result)

Skipping training for Vanilla_PPO at 100000 steps (already saved)
Skipping training for Vanilla_PPO at 500000 steps (already saved)
Skipping training for Vanilla_PPO at 1000000 steps (already saved)
Skipping training for Vanilla_PPO at 1500000 steps (already saved)
Skipping training for Vanilla_PPO at 2000000 steps (already saved)
Training Vanilla_PPO to 3000000 steps...
[Step 3] Reward: -10.00
[Step 5] Reward: 15.00
[Step 9] Reward: 15.65
[Step 10] Reward: 14.99
[Step 13] Reward: 15.65
[Step 16] Reward: -10.01
[Step 18] Reward: 15.64
[Step 21] Reward: 14.99
[Step 22] Reward: -9.34
[Step 23] Reward: -10.01
[Step 24] Reward: -9.35
[Step 26] Reward: 16.41
[Step 28] Reward: -9.38
[Step 29] Reward: 16.39
[Step 30] Reward: 15.63
[Step 32] Reward: 14.99
[Step 33] Reward: 15.66
[Step 34] Reward: -8.54
[Step 35] Reward: 17.42
[Step 36] Reward: 16.48
[Step 38] Reward: -7.72
[Step 40] Reward: -8.59
[Step 41] Reward: 17.34
[Step 42] Reward: 16.43
[Step 45] Reward: 15.65
[Step 46] Reward: -8.56
[S

KeyboardInterrupt: 

In [ ]:
for result in all_results:
    final = result["portfolio"][-1]
    print(f"{result['name']}: Final Value = £{final:,.2f}")
    plot_actions(result)
    print("Actions taken:", set(model["actions"]))
    print("Action counts:", np.bincount(model["actions"]))

In [ ]:

for model in all_results:
    print(f"Model: {model['name']}, final portfolio value: £{model['portfolio'][-1]:,.2f}")
    plt.figure(figsize=(12, 5))
    plt.plot(model["portfolio"], label="Portfolio Value")
    plt.title(f"Portfolio Value Over Time – {model['name']}")
    plt.xlabel("Step")
    plt.ylabel("£ Value")
    plt.grid(True)
    plt.legend()
    plt.show()

    unique_vals = set(model["portfolio"])
    print(f"📉 Unique portfolio values: {len(unique_vals)}")    
    print("Min:", min(model['portfolio']))
    print("Max:", max(model['portfolio']))

In [ ]:
from stable_baselines3 import PPO
import torch

def summarize_model(model_path, test_data, env_class, name):
    model = PPO.load(model_path)
    env = env_class(test_data.copy())
    obs, _ = env.reset()
    
    done = False
    steps = 0
    total_entropy = 0

    while not done:
        obs_tensor = torch.tensor([obs], dtype=torch.float32).to(model.device)
        dist = model.policy.get_distribution(obs_tensor)
        entropy = dist.entropy().item()

        action, _ = model.predict(obs, deterministic=True)
        obs, reward, done, truncated, info = env.step(action)

        total_entropy += entropy
        steps += 1

    final_value = env.cash + env.holdings * env.data.iloc[env.current_step]['Close']
    avg_entropy = total_entropy / steps

    print(f"📦 {name}")
    print(f"  🔢 Steps taken: {steps}")
    print(f"  💰 Final portfolio value: £{final_value:,.2f}")
    print(f"  🔀 Avg entropy: {avg_entropy:.4f}")
    print("-" * 40)

for model in all_results:
    summarize_model("./checkpoints/Agressive_model_2000000_steps.zip", training_data, StockTradingEnv, "Aggressive 2M")

In [ ]:
test_env = StockTradingEnv(testing_data.copy())
obs, _ = test_env.reset()
done = False

while not done:
    action = 1  # Always buy
    obs, reward, done, truncated, info = test_env.step(action)

print("Final value (always buy):", test_env.cash + test_env.holdings * test_env.data.iloc[test_env.current_step]['Close'])

Final value (always buy): 8772.208556537631
